In [1]:
from pathlib import Path

# Resolve paths from the repository root when launched here or from notebooks/.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'data' / 'raw').is_dir() and (path / 'notebooks').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the project root or notebooks/ folder.')

CLEAN_DIR = PROJECT_ROOT / 'data' / 'clean'
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

import os
os.getcwd()

'C:\\Users\\ewuzi\\Downloads\\Project\\notebooks'

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text

In [3]:
# Credentials are read from the repository-root .env, which is gitignored. Copy .env.example to .env
# and set PG_URL there. No password appears anywhere in this notebook.
import os

from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / '.env')
ENGINE_URL = os.environ["PG_URL"]
engine = create_engine(ENGINE_URL)

In [4]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT current_database();"))
    print(result.fetchone())

('swedish_electricity',)


In [5]:
import os

[file.name for file in sorted(CLEAN_DIR.glob('*.csv'))]

['clean_flows_hourly.csv',
 'clean_generation_hourly.csv',
 'clean_prices_hourly.csv',
 'clean_scb_monthly.csv',
 'clean_temperature_hourly.csv',
 'panel_hourly.csv',
 'panel_monthly.csv']

In [6]:
prices = pd.read_csv(CLEAN_DIR / "clean_prices_hourly.csv")
temperature = pd.read_csv(CLEAN_DIR / "clean_temperature_hourly.csv")
flows = pd.read_csv(CLEAN_DIR / "clean_flows_hourly.csv")
generation = pd.read_csv(CLEAN_DIR / "clean_generation_hourly.csv")
scb = pd.read_csv(CLEAN_DIR / "clean_scb_monthly.csv")

In [7]:
prices["ts_utc"] = pd.to_datetime(prices["ts_utc"], utc=True)
temperature["ts_utc"] = pd.to_datetime(temperature["ts_utc"], utc=True)
flows["ts_utc"] = pd.to_datetime(flows["ts_utc"], utc=True)
generation["ts_utc"] = pd.to_datetime(generation["ts_utc"], utc=True)

scb["month"] = pd.to_datetime(scb["month"])

In [8]:
prices.to_sql("prices_hourly", engine, if_exists="replace", index=False)
temperature.to_sql("temperature_hourly", engine, if_exists="replace", index=False)
flows.to_sql("flows_hourly", engine, if_exists="replace", index=False)
generation.to_sql("generation_hourly", engine, if_exists="replace", index=False)
scb.to_sql("scb_monthly", engine, if_exists="replace", index=False)

print("All 5 tables loaded successfully.")

All 5 tables loaded successfully.


In [9]:
with engine.begin() as conn:

    # Primary keys
    conn.execute(text("""
        ALTER TABLE prices_hourly
        ADD CONSTRAINT pk_prices_hourly PRIMARY KEY (ts_utc, zone);
    """))

    conn.execute(text("""
        ALTER TABLE temperature_hourly
        ADD CONSTRAINT pk_temperature_hourly PRIMARY KEY (ts_utc, zone);
    """))

    conn.execute(text("""
        ALTER TABLE flows_hourly
        ADD CONSTRAINT pk_flows_hourly PRIMARY KEY (ts_utc);
    """))

    conn.execute(text("""
        ALTER TABLE generation_hourly
        ADD CONSTRAINT pk_generation_hourly PRIMARY KEY (ts_utc);
    """))

    conn.execute(text("""
        ALTER TABLE scb_monthly
        ADD CONSTRAINT pk_scb_monthly PRIMARY KEY (category, zone, month);
    """))

    # Indexes for zone-based joins/filtering
    conn.execute(text("""
        CREATE INDEX idx_prices_zone
        ON prices_hourly (zone);
    """))

    conn.execute(text("""
        CREATE INDEX idx_temperature_zone
        ON temperature_hourly (zone);
    """))

    conn.execute(text("""
        CREATE INDEX idx_scb_zone
        ON scb_monthly (zone);
    """))

print("Primary keys and indexes created successfully.")

Primary keys and indexes created successfully.


In [10]:
query = """
SELECT 'prices_hourly' AS table_name, COUNT(*) AS rows FROM prices_hourly
UNION ALL
SELECT 'temperature_hourly', COUNT(*) FROM temperature_hourly
UNION ALL
SELECT 'flows_hourly', COUNT(*) FROM flows_hourly
UNION ALL
SELECT 'generation_hourly', COUNT(*) FROM generation_hourly
UNION ALL
SELECT 'scb_monthly', COUNT(*) FROM scb_monthly;
"""

pd.read_sql(query, engine)

,table_name,rows
0,scb_monthly,2880
1,flows_hourly,43824
2,generation_hourly,43824
3,prices_hourly,171959
4,temperature_hourly,174813
